# 2 - SILVER: 
## Dados padronizados, limpos e enriquecidos

>> RAW (CSV original)
        │
        ▼
>> BRONZE (Parquet 1:1 com o CSV)
        │
        ▼
> SILVER (dados padronizados, limpos e enriquecidos)
        │
        ▼
GOLD (indicadores e agregações)

## 2.0 - Bibliotecas

In [1]:
from dotenv import load_dotenv
from datetime import datetime
from src.data.utils import *
from src.data.quality import *
from src.data.metadata import *

load_dotenv()
session = iniciar_cessao_aws()
s3 = session.client("s3")
BUCKET = os.getenv("BUCKET_NAME")



/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


# 2.1 - ETL - Bronze para Silver

## 2.1.1 - Def Tabelas e Mapas

In [ ]:
TABELAS = [
    "TS_ALUNO",
    "TS_ESTADO",
    "TS_MUNICIPIO",
    "TS_ITEM",
    "RESULTADOS_METAS",
    "RESULTADOS_METAS_MUNICIPIO"
]

# anos de interesse - poderia pegar diretamente do nome do arquivo, mas para fins de teste, vamos filtrar por ano
ANOS = [2023, 2024, 2025]

# Mapas para mapeamento de códigos para descrições

MAPA_REDES = {
    0: 'TOTAL',
    1: 'FEDERAL',
    2: 'ESTADUAL',
    3: 'MUNICIPAL',
    4: 'PRIVADA',
    5: 'PUBLICA_EST_MUN',
    6: 'PUBLICA_TOTAL'
}


MAPA_DISCIPLINA = {
    1: "Língua Portuguesa"
}

MAPA_SERIE = {
    2: "2º Ano Do Ensino Fundamental"
}

#--------------------------------
MAPAS = {
    "TP_REDE": MAPA_REDES,
    "TP_DEPENDENCIA": MAPA_REDES, ##  TP_DEPENDENCIA uses the same mapping as TP_REDE, TP_DEPENDENCIA is a subset of TP_REDE
    "TP_DISCIPLINA": MAPA_DISCIPLINA,
    "TP_SERIE": MAPA_SERIE,
}

## 2.1.2 - Funções

In [14]:


paginator = s3.get_paginator("list_objects_v2")

returned = paginator.paginate(
    Bucket=BUCKET,
    Prefix="bronze/",
    Delimiter="/"
)

In [19]:
# listar pastas
def listar_pastas_s3(
    s3_client,
    bucket: str,
    prefixo: str = "silver/"
):
    """
    Lista as subpastas de um prefixo.
    """

    resposta = s3_client.list_objects_v2(
        Bucket=bucket,
        Prefix=prefixo,
        Delimiter="/"
    )

    if "CommonPrefixes" not in resposta:
        return []

    pastas = []

    for pasta in resposta["CommonPrefixes"]:
        print(pasta["Prefix"])
        pastas.append(pasta["Prefix"])

    return pastas

# listar arquivos
def listar_arquivos_s3(
    s3_client,
    bucket: str,
    prefixo: str = "bronze/"
):
    """
    Lista todos os arquivos de um prefixo do S3.
    """

    resposta = s3_client.list_objects_v2(
        Bucket=bucket,
        Prefix=prefixo
    )

    if "Contents" not in resposta:
        print("Nenhum arquivo encontrado.")
        return []

    arquivos = []

    for obj in resposta["Contents"]:
        print(obj["Key"])
        arquivos.append(obj["Key"])

    return arquivos

# mapa
def aplicar_mapas(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Aplica automaticamente os mapas às colunas existentes.
    """

    for coluna, mapa in MAPAS.items():

        if coluna not in df.columns:
            if verbose:
                print(f"⚪ {coluna}: não encontrada.")
            continue

        nova_coluna = f"DESC_{coluna[3:]}"

        df[nova_coluna] = (
            pd.to_numeric(df[coluna], errors="coerce")
            .map(mapa)
        )

        if verbose:
            print(f"✅ {coluna} → {nova_coluna}")

    return df

#conversor de colunas IDs para inteiro
def converter_colunas_int(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    """
    Converte as colunas informadas para inteiro (Int64),
    ignorando colunas inexistentes.
    """

    for coluna in colunas:

        if coluna in df.columns:

            df[coluna] = (
                pd.to_numeric(
                    df[coluna],
                    errors="coerce"
                )
                .astype("Int64")
            )

    return df

## 2.1.3 - ETL
Extração: 
    - Extrai da camada Bronze
Tratamento: 
    - Aplica Data Quality
    - Aplica Mapas
Load: 
    - Salva na camada Silver

In [ ]:
#colunas que devem ser convertidas para inteiro

COLUNAS_INT = [
    "NU_ANO_AVALIACAO",
    "CO_UF",
    "ID_ALUNO",
    "TP_SERIE",
    "ID_ESCOLA",
    "TP_DEPENDENCIA",
    "CO_MUNICIPIO",
    "IN_PRESENCA_LP",
    "ID_TIPO_REDE"
]

In [16]:


relatorios = []
catalogos = []

for ano in ANOS:

    for tabela in TABELAS:

        print(f"\n{'='*60}")
        print(f"Processando {ano} - {tabela}")
        print(f"{'='*60}")

        try:

            # -----------------------------
            # Bronze
            # -----------------------------

            print(f"Carregando Bronze: {ano} - {tabela}")
            df_bronze = carregar_parquet_s3(
                s3_client=s3,
                bucket=BUCKET,
                ano=ano,
                nome_tabela=tabela,
                camada="bronze"
            )

            print(f"Carregando Dicionário: {ano} - {tabela}")
            dic = carregar_parquet_s3(
                s3_client=s3,
                bucket=BUCKET,
                ano=ano,
                nome_tabela=tabela,
                ler_dicionario=True,
                camada="bronze"
            )

            print(f"✔ Carga concluída: {ano} - {tabela} - {len(df_bronze):,} linhas e {len(df_bronze.columns):,} colunas.")

            print(f"Aplicando Data Quality: {ano} - {tabela}")
            
            # Faz uma cópia para comparação
            df_silver = df_bronze.copy()

            # -----------------------------
            # Data Quality
            # -----------------------------
          
            df_silver = aplicar_data_quality(df_silver, dic)
            print(f"✔ Data Quality concluída: {ano} - {tabela} - {len(df_silver):,} linhas e {len(df_silver.columns):,} colunas.")
            
            # -----------------------------
            # Mapas
            # -----------------------------

            print(f"Aplicando Mapas: {ano} - {tabela}")
            df_silver = aplicar_mapas(df_silver)
            print(f"✔ Mapas concluídos: {ano} - {tabela} - {len(df_silver):,} linhas e {len(df_silver.columns):,} colunas.")

            # -----------------------------
            # Salva Silver
            # -----------------------------
            print(f"Salvando Silver: {ano} - {tabela}")
            salvar_parquet_s3(
                s3_client=s3,
                bucket=BUCKET,
                df=df_silver,
                ano=ano,
                tabela=tabela,
                camada="silver"
            )

            # -----------------------------
            # Relatório
            # -----------------------------
            print(f"Gerando Relatório: {ano} - {tabela}")
            relatorio = gerar_relatorio_data_quality(
                bronze=df_bronze,
                silver=df_silver,
                ano=ano,
                tabela=tabela
            )

            relatorios.append(relatorio)

            # -----------------------------
            # Catálogo
            # -----------------------------
            print(f"Gerando Catálogo: {ano} - {tabela}")
            catalogo = gerar_catalogo_processamento(
                df=df_silver,
                ano=ano,
                tabela=tabela,
                camada="silver"
            )

            catalogos.append(catalogo)

            print("✔ Processamento concluído.")

        except Exception as e:

            print(f"Erro em {ano} - {tabela}")
            print(e)


Processando 2023 - TS_ALUNO
Carregando Bronze: 2023 - TS_ALUNO
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2023/dados/TS_ALUNO.parquet
Carregando Dicionário: 2023 - TS_ALUNO
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2023/dicionario/dicionario_TS_ALUNO.parquet
✔ Carga concluída: 2023 - TS_ALUNO - 1,747,439 linhas e 15 colunas.
Aplicando Data Quality: 2023 - TS_ALUNO
Iniciando Data Quality para 1,747,439 linhas e 15 colunas.
Após remover vazias : 1,747,439
Após remover duplicados : 1,747,439
Nulos tratados : 489260
Textos padronizados : 0
Data Quality concluída.
✔ Data Quality concluída: 2023 - TS_ALUNO - 1,747,439 linhas e 15 colunas.
Aplicando Mapas: 2023 - TS_ALUNO
⚪ TP_REDE: não encontrada.
✅ TP_DEPENDENCIA → DESC_DEPENDENCIA
⚪ TP_DISCIPLINA: não encontrada.
✅ TP_SERIE → DESC_SERIE
✔ Mapas concluídos: 2023 - TS_ALUNO - 1,747,439 linhas e 17 colunas.
Salvando Silver: 2023 - TS_ALUNO
Arquivo salvo: s3://fiap-postech-challenge-datascience-002/silve

In [ ]:
df["ID_ESCOLA"] = (
    pd.to_numeric(df["ID_ESCOLA"], errors="coerce")
      .astype("Int64")
)

## 2.1.4 - Guardando os Metadados

In [17]:
df_relatorio = pd.concat(relatorios, ignore_index=True)

df_catalogo = pd.concat(catalogos, ignore_index=True)

In [18]:
salvar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    df=df_relatorio,
    ano=datetime.now().year,
    tabela="DATA_QUALITY",
    camada="silver/metadata"
)

salvar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    df=df_catalogo,
    ano=datetime.now().year,
    tabela="CATALOGO_PROCESSAMENTO",
    camada="silver/metadata"
)

Arquivo salvo: s3://fiap-postech-challenge-datascience-002/silver/metadata/ano=2026/dados/DATA_QUALITY.parquet
Arquivo salvo: s3://fiap-postech-challenge-datascience-002/silver/metadata/ano=2026/dados/CATALOGO_PROCESSAMENTO.parquet


'silver/metadata/ano=2026/dados/CATALOGO_PROCESSAMENTO.parquet'

## testes e validaçoes

### Listar arquivos

In [10]:
# listar arquivos da camada bronze
arquivos = listar_arquivos_s3(
    s3,
    BUCKET,
    "bronze/"
)

# listar arquivos da camada silver
arquivos = listar_arquivos_s3(
    s3,
    BUCKET,
    "silver/"
)



bronze/ano=2023/dados/RESULTADOS_METAS.parquet
bronze/ano=2023/dados/RESULTADOS_METAS_MUNICIPIO.parquet
bronze/ano=2023/dados/TS_ALUNO.parquet
bronze/ano=2023/dados/TS_ESTADO.parquet
bronze/ano=2023/dados/TS_ITEM.parquet
bronze/ano=2023/dados/TS_MUNICIPIO.parquet
bronze/ano=2023/dicionario/dicionario_RESULTADOS_METAS.parquet
bronze/ano=2023/dicionario/dicionario_RESULTADOS_METAS_MUNICIPIO.parquet
bronze/ano=2023/dicionario/dicionario_TS_ALUNO.parquet
bronze/ano=2023/dicionario/dicionario_TS_ESTADO.parquet
bronze/ano=2023/dicionario/dicionario_TS_ITEM.parquet
bronze/ano=2023/dicionario/dicionario_TS_MUNICIPIO.parquet
bronze/ano=2024/dados/RESULTADOS_METAS.parquet
bronze/ano=2024/dados/RESULTADOS_METAS_MUNICIPIO.parquet
bronze/ano=2024/dados/TS_ALUNO.parquet
bronze/ano=2024/dados/TS_ESTADO.parquet
bronze/ano=2024/dados/TS_ITEM.parquet
bronze/ano=2024/dados/TS_MUNICIPIO.parquet
bronze/ano=2024/dicionario/dicionario_RESULTADOS_METAS.parquet
bronze/ano=2024/dicionario/dicionario_RESULTADOS_

In [12]:
#carrega da camada silver - mostrar os resultados 

df_silver = carregar_parquet_s3(
    s3_client=s3,
    bucket=BUCKET,
    ano=2025,
    nome_tabela="TS_ALUNO",
    camada="silver"
).head(100)

df_silver.head() 


Lendo: s3://fiap-postech-challenge-datascience-002/silver/ano=2025/dados/TS_ALUNO.parquet


,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,...,TX_RESPOSTA_BLOCO_3,TX_GABARITO_BLOCO_3,CO_BLOCO_4,TX_RESPOSTA_BLOCO_4,TX_GABARITO_BLOCO_4,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO,DESC_DEPENDENCIA,DESC_SERIE
0,2025,11,RO,11000677,2,60000060.0,3.0,1100023.0,Ariquemes,1,...,BBCD,ABCB,NaN,<NA>,<NA>,1.5,781.776332,1,Municipal,2º Ano Do Ensino Fundamental
1,2025,11,RO,11000678,2,60000060.0,3.0,1100023.0,Ariquemes,1,...,DACB,ABCB,NaN,<NA>,<NA>,1.5,717.877346,0,Municipal,2º Ano Do Ensino Fundamental
2,2025,11,RO,11000679,2,60000060.0,3.0,1100023.0,Ariquemes,1,...,BBBB,ABBD,NaN,<NA>,<NA>,1.5,657.836805,0,Municipal,2º Ano Do Ensino Fundamental
3,2025,11,RO,11000680,2,60000060.0,3.0,1100023.0,Ariquemes,1,...,CABA,ABDA,NaN,<NA>,<NA>,1.5,765.290294,1,Municipal,2º Ano Do Ensino Fundamental
4,2025,11,RO,11000681,2,60000060.0,3.0,1100023.0,Ariquemes,1,...,CABA,CABA,NaN,<NA>,<NA>,1.5,835.506362,1,Municipal,2º Ano Do Ensino Fundamental


In [ ]:
#carrega da camada silver e mostra os resultados 
for ano in ANOS:

    for tabela in TABELAS:

        print(f"Processando {ano} - {tabela}")

        df = carregar_parquet_s3(
            s3_client=s3,
            bucket=BUCKET,
            ano=ano,
            nome_tabela=tabela
            camada="silver"
        ).head(100)


Processando 2023 - TS_ALUNO
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2023/dados/TS_ALUNO.parquet
Processando 2023 - TS_ESTADO
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2023/dados/TS_ESTADO.parquet
Processando 2023 - TS_ITEM
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2023/dados/TS_ITEM.parquet
Processando 2023 - TS_MUNICIPIO
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2023/dados/TS_MUNICIPIO.parquet
Processando 2023 - RESULTADOS_METAS
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2023/dados/RESULTADOS_METAS.parquet
Processando 2024 - TS_ALUNO
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2024/dados/TS_ALUNO.parquet
Processando 2024 - TS_ESTADO
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2024/dados/TS_ESTADO.parquet
Processando 2024 - TS_ITEM
Lendo: s3://fiap-postech-challenge-datascience-002/bronze/ano=2024/dados/TS_ITEM.parquet
Processando 2024 - TS_MUNICIPIO


,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
1,2023,11,RO,11008695,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,714.314857,0
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
3,2023,11,RO,11008682,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,759.206313,1
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,0,19,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2023,11,RO,11009616,2,60000014,3,1100205,Porto Velho,1,1,2,1.092938,767.646953,1
996,2023,11,RO,11009526,2,60000014,3,1100205,Porto Velho,1,1,19,1.037331,808.248669,1
997,2023,11,RO,11009472,2,60000014,3,1100205,Porto Velho,1,1,17,1.047399,730.725020,0
998,2023,11,RO,11009483,2,60000014,3,1100205,Porto Velho,1,1,17,1.047399,785.093300,1


# FIM!

## Funções2 - OLD
algumas estão no src.data.utils.py

In [3]:
import io

session = iniciar_cessao_aws()
s3 = session.client('s3')
BUCKET_NAME = os.getenv("BUCKET_NAME")
ARQUIVO = 'RESULTADOS_METAS'
PASTA = f"bronze/ano={ano}/dados/{ARQUIVO}"

response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PASTA)
arquivos = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.parquet')]

print(f"Total de arquivos encontrados: {len(arquivos)}")
for f in arquivos[:5]:
    print(f"  - {f}")

# Leitura dos arquivos em um DataFrame
dfs = []
for chave in arquivos:
    obj = s3.get_object(Bucket=BUCKET_NAME, Key=chave)
    df_part = pd.read_parquet(io.BytesIO(obj['Body'].read()))
    dfs.append(df_part)

dataframe = pd.concat(dfs, ignore_index=True)

print(f"\nTotal de linhas: {len(dataframe):,}")
print(f"Colunas: {dataframe.shape[1]}")
dataframe.head()
    

Total de arquivos encontrados: 2
  - bronze/ano=2025/dados/RESULTADOS_METAS.parquet
  - bronze/ano=2025/dados/RESULTADOS_METAS_MUNICIPIO.parquet

Total de linhas: 5,500
Colunas: 22


,ANO,CD_UF,SIGLA_UF,NOME_UF,REDE,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,PC_ALUNO_ALFABETIZADO_2025,META_FINAL_2024,META_FINAL_2025,...,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,PC_AVALIADOS_LP,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,NO_TP_REDE,CO_NIVEL_ALFABETIZACAO
0,2025.0,NaN,NaN,Brasil,PÚBLICA,56.0,59.0,66.0,60.0,64.0,...,74.0,77.0,> 80,89.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025.0,12.0,AC,Acre,PÚBLICA,NaN,51.0,68.0,NaN,57.0,...,72.0,76.0,> 80,82.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2025.0,27.0,AL,Alagoas,PÚBLICA,44.0,49.0,64.0,50.0,56.0,...,72.0,76.0,> 80,94.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025.0,13.0,AM,Amazonas,PÚBLICA,52.0,49.0,57.0,57.0,61.0,...,73.0,77.0,> 80,86.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025.0,16.0,AP,Amapá,PÚBLICA,42.0,47.0,60.0,48.0,54.0,...,71.0,76.0,> 80,88.0,NaN,NaN,NaN,NaN,NaN,NaN


ano = 2025
raw_municipio = carregar_parquet_local(ano, 'TS_MUNICIPIO')
raw_municipio

## Criada uma função para executar o pipeline automaticamente a partir da camada bronze

In [4]:
def processar_camada_silver(ano: int | str) -> pd.DataFrame:
    """
    Processa e integra as bases da camada Bronze, gerando o DataFrame final
    da camada Silver enriquecido com dados de município e UF para o ano informado.
    """
    # 1. Carrega automaticamente as bases Bronze locais
    print(f"\n[INFO] Carregando dados do ano {ano}...")
    raw_alunos = carregar_parquet_local(ano, 'TS_ALUNO')
    raw_municipio = carregar_parquet_local(ano, 'TS_MUNICIPIO')
    raw_uf = carregar_parquet_local(ano, 'TS_ESTADO')
    
    MAPA_REDES = {
        0: 'TOTAL',
        1: 'FEDERAL',
        2: 'ESTADUAL',
        3: 'MUNICIPAL',
        4: 'PRIVADA',
        5: 'PUBLICA_EST_MUN',
        6: 'PUBLICA_TOTAL'
    }
    

In [5]:

    
    # === A. PREPARAÇÃO DO MUNICÍPIO =========================================
    print(f"[INFO] Processando dimensões do Município para {ano}...")
    CHAVES_MUN = ['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE']
    REDES_MUN = list(MAPA_REDES.values())
    
    municipio_prep = (
        raw_municipio[CHAVES_MUN + ['ID_TIPO_REDE'] + METRICAS]
        .assign(REDE_NOME=lambda df: df['ID_TIPO_REDE'].map(MAPA_REDES))
        .query('REDE_NOME.notna()')   
        .drop(columns='ID_TIPO_REDE')
    )
    
    municipio_pivot = municipio_prep.pivot(index=CHAVES_MUN, columns='REDE_NOME', values=METRICAS)
    municipio_pivot.columns = [f"{metrica}_{rede}_MUNICIPIO" for metrica, rede in municipio_pivot.columns]
    municipio_dim = municipio_pivot.reset_index()
    
    # Garante consistência de colunas (trata anos sem ID 0)
    colunas_esperadas_mun = [f"{metrica}_{rede}_MUNICIPIO" for metrica in METRICAS for rede in REDES_MUN]
    municipio_dim = municipio_dim.reindex(columns=CHAVES_MUN + colunas_esperadas_mun)
    
    # Fallback TOTAL -> PUBLICA_EST_MUN
    for metrica in METRICAS:
        col_total = f"{metrica}_TOTAL_MUNICIPIO"
        col_publica = f"{metrica}_PUBLICA_EST_MUN_MUNICIPIO"
        municipio_dim[col_total] = municipio_dim[col_total].fillna(municipio_dim[col_publica])

    # === B. PREPARAÇÃO DA UF ===============================================
    print(f"[INFO] Processando dimensões da UF para {ano}...")
    CHAVES_UF = ['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE']
    REDES_UF = list(MAPA_REDES.values())
    
    uf_prep = (
        raw_uf[CHAVES_UF + ['ID_TIPO_REDE'] + METRICAS]
        .assign(REDE_NOME=lambda df: df['ID_TIPO_REDE'].map(MAPA_REDES))
        .query('REDE_NOME.notna()')
        .drop(columns='ID_TIPO_REDE')
    )
    
    uf_pivot = uf_prep.pivot(index=CHAVES_UF, columns='REDE_NOME', values=METRICAS)
    uf_pivot.columns = [f"{metrica}_{rede}_UF" for metrica, rede in uf_pivot.columns]
    uf_dim = uf_pivot.reset_index()
    
    # Garante consistência de colunas da UF
    colunas_esperadas_uf = [f"{metrica}_{rede}_UF" for metrica in METRICAS for rede in REDES_UF]
    uf_dim = uf_dim.reindex(columns=CHAVES_UF + colunas_esperadas_uf)
    
    # Fallback TOTAL -> PUBLICA_EST_MUN
    for metrica in METRICAS:
        col_total = f"{metrica}_TOTAL_UF"
        col_publica = f"{metrica}_PUBLICA_EST_MUN_UF"
        uf_dim[col_total] = uf_dim[col_total].fillna(uf_dim[col_publica])

    # === C. JUNCÕES (MERGE) =================================================
    print(f"[INFO] Mesclando dados na tabela de alunos...")
    df_silver = pd.merge(
        raw_alunos,
        municipio_dim,
        on=['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE'],
        how='left'
    )
    
    df_silver = pd.merge(
        df_silver,
        uf_dim,
        on=['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE'],
        how='left'
    )

    #  CRIAÇÃO DE COLUNAS DERIVADAS E LIMPEZA 
    print(f"[INFO] Calculando desvios e limpando colunas vazias...")
    df_silver['DESVIO_MEDIA_MUNICIPIO'] = df_silver['VL_PROFICIENCIA_LP'] - df_silver['VL_MEDIA_LP_TOTAL_MUNICIPIO']
    df_silver['DESVIO_MEDIA_UF'] = df_silver['VL_PROFICIENCIA_LP'] - df_silver['VL_MEDIA_LP_TOTAL_UF']
    
    # Dropa as colunas de redes nulas e as respostas das provas 
    colunas_para_remover = [
        # 1. Redes inativas do Município (100% nulas)
        'VL_MEDIA_LP_FEDERAL_MUNICIPIO', 'PC_ALUNO_ALFABETIZADO_FEDERAL_MUNICIPIO',
        'VL_MEDIA_LP_PRIVADA_MUNICIPIO', 'PC_ALUNO_ALFABETIZADO_PRIVADA_MUNICIPIO',
        'VL_MEDIA_LP_PUBLICA_TOTAL_MUNICIPIO', 'PC_ALUNO_ALFABETIZADO_PUBLICA_TOTAL_MUNICIPIO',
        
        # 2. Redes inativas da UF (100% nulas)
        'VL_MEDIA_LP_FEDERAL_UF', 'PC_ALUNO_ALFABETIZADO_FEDERAL_UF',
        'VL_MEDIA_LP_PRIVADA_UF', 'PC_ALUNO_ALFABETIZADO_PRIVADA_UF',
        'VL_MEDIA_LP_PUBLICA_TOTAL_UF', 'PC_ALUNO_ALFABETIZADO_PUBLICA_TOTAL_UF',
        
        # 3. Colunas vazias de Prova (Bloco 4 não foi utilizado nas avaliações)
        'CO_BLOCO_4', 'TX_RESPOSTA_BLOCO_4', 'TX_GABARITO_BLOCO_4'
    ]
    df_silver = df_silver.drop(columns=colunas_para_remover, errors='ignore')

    print(f"[INFO] Ano {ano} processado com sucesso!")
    return df_silver


[INFO] Processando dimensões do Município para 2025...


NameError: name 'MAPA_REDES' is not defined

In [ ]:
# Lista todas as colunas que possuem a palavra 'BLOCO' no nome
[col for col in raw_alunos.columns if 'BLOCO' in col]


['CO_BLOCO_1',
 'TX_RESPOSTA_BLOCO_1',
 'TX_GABARITO_BLOCO_1',
 'CO_BLOCO_2',
 'TX_RESPOSTA_BLOCO_2',
 'TX_GABARITO_BLOCO_2',
 'CO_BLOCO_3',
 'TX_RESPOSTA_BLOCO_3',
 'TX_GABARITO_BLOCO_3',
 'CO_BLOCO_4',
 'TX_RESPOSTA_BLOCO_4',
 'TX_GABARITO_BLOCO_4']

In [ ]:
raw_alunos.head(5).T

,0,1,2,3,4
NU_ANO_AVALIACAO,2023,2023,2023,2023,2023
CO_UF,11,11,11,11,11
SG_UF,RO,RO,RO,RO,RO
ID_ALUNO,11008701,11008695,11008687,11008682,11008729
TP_SERIE,2,2,2,2,2
ID_ESCOLA,60000001,60000001,60000001,60000001,60000001
TP_DEPENDENCIA,3,3,3,3,3
CO_MUNICIPIO,1100205,1100205,1100205,1100205,1100205
NO_MUNICIPIO,Porto Velho,Porto Velho,Porto Velho,Porto Velho,Porto Velho
IN_PRESENCA_LP,0,1,0,1,0


In [ ]:
#  Pivot da tabela munipicio

MAPA_REDES = {
    0: 'TOTAL',
    1: 'FEDERAL',
    2: 'ESTADUAL',
    3: 'MUNICIPAL',
    4: 'PRIVADA',
    5: 'PUBLICA_EST_MUN',
    6: 'PUBLICA_TOTAL'
}


METRICAS = ['VL_MEDIA_LP', 'PC_ALUNO_ALFABETIZADO']
CHAVES    = ['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE']
REDES     = list(MAPA_REDES.values())  # ordem esperada de colunas

# Filtra, mapeia as redes
municipio_prep = (
    raw_municipio[CHAVES + ['ID_TIPO_REDE'] + METRICAS]
    .assign(REDE_NOME=lambda df: df['ID_TIPO_REDE'].map(MAPA_REDES))
    .query('REDE_NOME.notna()')   
    .drop(columns='ID_TIPO_REDE')
)

# Pivot: uma coluna por (métrica × rede) 
municipio_pivot = municipio_prep.pivot(
    index=CHAVES,
    columns='REDE_NOME',
    values=METRICAS
)

# Achata MultiIndex → "VL_MEDIA_LP_ESTADUAL_MUNICIPIO" etc. ──────────
municipio_pivot.columns = [
    f"{metrica}_{rede}_MUNICIPIO"
    for metrica, rede in municipio_pivot.columns
]
municipio_dim = municipio_pivot.reset_index()

# Garante todas as colunas esperadas (resiliente a anos incompletos) ──
colunas_esperadas = [
    f"{metrica}_{rede}_MUNICIPIO"
    for metrica in METRICAS
    for rede in REDES
]
municipio_dim = municipio_dim.reindex(
    columns=CHAVES + colunas_esperadas  # ordem determinística
)

# ── 5. Fallback: TOTAL ausente → usa PUBLICA_EST_MUN ─────────────────────
for metrica in METRICAS:
    col_total   = f"{metrica}_TOTAL_MUNICIPIO"
    col_publica = f"{metrica}_PUBLICA_EST_MUN_MUNICIPIO"
    municipio_dim[col_total] = municipio_dim[col_total].fillna(
        municipio_dim[col_publica]
    )

In [ ]:
#Pivot da tabela UF

uf_prep = raw_uf[
    ['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE', 'ID_TIPO_REDE', 'VL_MEDIA_LP', 'PC_ALUNO_ALFABETIZADO']
].copy()

#utilizar o dicionario para converter os IDs em  texto
uf_prep['REDE_NOME'] = uf_prep['ID_TIPO_REDE'].map(MAPA_REDES)

# Criar um pivot da tabela 
uf_pivot = uf_prep.pivot(
    index=['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE'],
    columns='REDE_NOME',
    values=['VL_MEDIA_LP', 'PC_ALUNO_ALFABETIZADO']
)

# Gerar MultiIndex das colunas
uf_pivot.columns = [f"{metrica}_{rede}_UF" for metrica, rede in uf_pivot.columns]
uf_dim = uf_pivot.reset_index()



uf_dim.head(3)


,NU_ANO_AVALIACAO,CO_UF,TP_SERIE,VL_MEDIA_LP_ESTADUAL_UF,VL_MEDIA_LP_MUNICIPAL_UF,VL_MEDIA_LP_PUBLICA_EST_MUN_UF,PC_ALUNO_ALFABETIZADO_ESTADUAL_UF,PC_ALUNO_ALFABETIZADO_MUNICIPAL_UF,PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_UF
0,2023,11,2,751.4731,760.1971,759.4357,58.65,65.17,64.60
1,2023,13,2,746.2058,733.6637,736.4687,62.63,49.20,52.20
2,2023,15,2,741.3548,732.6697,733.2877,57.07,47.71,48.38


In [ ]:
# Merge dos alunos com munipio e UF
df_silver = pd.merge(
    raw_alunos,
    municipio_dim,
    on=['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE'],
    how='left'
)

# ── Etapa 4: + UF 
df_silver = pd.merge(
    df_silver,
    uf_dim,
    on=['NU_ANO_AVALIACAO', 'CO_UF', 'TP_SERIE'],
    how='left'
)



In [ ]:
df_silver.head(5)

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,...,PC_ALUNO_ALFABETIZADO_MUNICIPAL_MUNICIPIO,PC_ALUNO_ALFABETIZADO_PRIVADA_MUNICIPIO,PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_MUNICIPIO,PC_ALUNO_ALFABETIZADO_PUBLICA_TOTAL_MUNICIPIO,VL_MEDIA_LP_ESTADUAL_UF,VL_MEDIA_LP_MUNICIPAL_UF,VL_MEDIA_LP_PUBLICA_EST_MUN_UF,PC_ALUNO_ALFABETIZADO_ESTADUAL_UF,PC_ALUNO_ALFABETIZADO_MUNICIPAL_UF,PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_UF
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6
1,2023,11,RO,11008695,2,60000001,3,1100205,Porto Velho,1,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6
3,2023,11,RO,11008682,2,60000001,3,1100205,Porto Velho,1,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,...,65.06,NaN,64.47,NaN,751.4731,760.1971,759.4357,58.65,65.17,64.6


In [ ]:
df_silver.shape

(1747439, 35)

In [ ]:
itens_na = pd.DataFrame(df_silver.isna().sum(), columns=['Qtd_Nulos'])
itens_na_filtrado = itens_na[itens_na['Qtd_Nulos'] > 0]

itens_na_filtrado.sort_values(by='Qtd_Nulos')



,Qtd_Nulos
PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_MUNICIPIO,446
VL_MEDIA_LP_TOTAL_MUNICIPIO,446
PC_ALUNO_ALFABETIZADO_TOTAL_MUNICIPIO,446
VL_MEDIA_LP_PUBLICA_EST_MUN_MUNICIPIO,446
PC_ALUNO_ALFABETIZADO_MUNICIPAL_MUNICIPIO,2413
VL_MEDIA_LP_MUNICIPAL_MUNICIPIO,2413
PC_ALUNO_ALFABETIZADO_ESTADUAL_UF,180056
VL_MEDIA_LP_ESTADUAL_UF,180056
VL_PESO_ALUNO_LP,244630
VL_PROFICIENCIA_LP,244630


Temos um grande numero de Nulos na base, isso se da porque algumas escolas nao participaram da prova, nao temos nada na esfera Federal (praticamente nao existem escolas federais de ensino fundamental no Brasil) 
as escolas privadas também nao participam da prova

**VL_MEDIA_LP_TOTAL_MUNICIPIO e PC_ALFAB_MUNICIPIO:** <br><br>
O
O que significa: Representa alunos que estudam em municípios extremamente pequenos cujos dados de médias foram ocultados pelo INEP por segredo estatístico (quando a cidade tem pouquíssimos alunos, o INEP não divulga as médias para não expor notas individuais). São perdas aceitáveis e muito pequenas (apenas 446 alunos no Brasil inteiro).

**VL_MEDIA_LP_MUNICIPAL_MUNICIPIO:**<br><br>
O que significa: Alunos que estudam em cidades onde não existem escolas municipais oferecendo o 2º ano do Ensino Fundamental. Nesses locais, toda a educação de alfabetização é concentrada na Rede Estadual.


**VL_PROFICIENCIA_LP e VL_PESO_ALUNO_LP**<br><br>
O que significa: Como analisamos anteriormente, são os alunos que faltaram no dia da prova ou tiveram suas avaliações anuladas/desconsideradas pelo INEP. Eles não possuem nota individual de proficiência.



**VL_MEDIA_LP_ESTADUAL_UF:** <br><br>
O que significa:
Ocorre principalmente no Distrito Federal (DF), onde não existe a divisão entre rede Estadual e Municipal (tudo é rede Distrital). Por conta disso, a categoria "Estadual" do DF fica inteiramente nula.
Estados onde a alfabetização foi 100% municipalizada e a secretaria estadual não gerencia nenhuma escola de 2º ano.


**VL_MEDIA_LP_ESTADUAL_MUNICIPIO**<br><br>
O que significa: Reflete o processo brasileiro de municipalização da alfabetização. Na imensa maioria das cidades do interior do país, o estado repassou a responsabilidade dos anos iniciais do ensino fundamental (1º ao 5º ano) inteiramente para as prefeituras. Como não existem escolas estaduais de 2º ano nesses municípios, essa coluna fica vazia para mais de 1 milhão de registros.



**VL_MEDIA_LP_FEDERAL_MUNICIPIO e VL_MEDIA_LP_PRIVADA_MUNICIPIO:** <br><br>
Ausência de Dados Federais e Privados  <br>
O que significa: Praticamente toda a base de alunos está nula nestas colunas porque a rede Federal e Privada não participam ou não possuem turmas de 2º ano do Ensino Fundamental registradas na avaliação na quase totalidade das cidades do Brasil.



In [ ]:
# Remove as colunas que estão 
colunas_para_remover = [
    'VL_MEDIA_LP_FEDERAL_MUNICIPIO',
    'PC_ALUNO_ALFABETIZADO_FEDERAL_MUNICIPIO',
    'VL_MEDIA_LP_PRIVADA_MUNICIPIO',
    'PC_ALUNO_ALFABETIZADO_PRIVADA_MUNICIPIO',
    'VL_MEDIA_LP_PUBLICA_TOTAL_MUNICIPIO',
    'PC_ALUNO_ALFABETIZADO_PUBLICA_TOTAL_MUNICIPIO'
]

df_silver = df_silver.drop(columns=colunas_para_remover, errors='ignore')


In [ ]:
df_silver['ANO_AVALIACAO'] = ano

In [ ]:
df_silver.head(5).T

,0,1,2,3,4
NU_ANO_AVALIACAO,2023,2023,2023,2023,2023
CO_UF,11,11,11,11,11
SG_UF,RO,RO,RO,RO,RO
ID_ALUNO,11008701,11008695,11008687,11008682,11008729
TP_SERIE,2,2,2,2,2
ID_ESCOLA,60000001,60000001,60000001,60000001,60000001
TP_DEPENDENCIA,3,3,3,3,3
CO_MUNICIPIO,1100205,1100205,1100205,1100205,1100205
NO_MUNICIPIO,Porto Velho,Porto Velho,Porto Velho,Porto Velho,Porto Velho
IN_PRESENCA_LP,0,1,0,1,0


In [ ]:
itens_na = pd.DataFrame(df_silver.isna().sum(), columns=['Qtd_Nulos'])
itens_na_filtrado = itens_na[itens_na['Qtd_Nulos'] > 0]

itens_na_filtrado.sort_values(by='Qtd_Nulos')



,Qtd_Nulos
VL_MEDIA_LP_TOTAL_MUNICIPIO,446
VL_MEDIA_LP_PUBLICA_EST_MUN_MUNICIPIO,446
PC_ALUNO_ALFABETIZADO_TOTAL_MUNICIPIO,446
PC_ALUNO_ALFABETIZADO_PUBLICA_EST_MUN_MUNICIPIO,446
VL_MEDIA_LP_MUNICIPAL_MUNICIPIO,2413
PC_ALUNO_ALFABETIZADO_MUNICIPAL_MUNICIPIO,2413
VL_MEDIA_LP_ESTADUAL_UF,180056
PC_ALUNO_ALFABETIZADO_ESTADUAL_UF,180056
VL_PESO_ALUNO_LP,244630
VL_PROFICIENCIA_LP,244630
